# Phase 4: Posterior Predictive Checks

This notebook performs posterior predictive checks (PPCs) using the fitted outputs from the earlier phases. It **does not refit** the Phase 1 mood model or the Phase 3 user-taste model. Instead, it loads the saved model artifacts and asks whether generated songs and generated listen events look like the real data.

The checks follow the Phase 4 project goal: compare fake-vs-real feature distributions, mood-frequency distributions, predicted listen probabilities, calibration, and listen-count distributions.

## 0  Imports and shared configuration

We use the same import style and random seed convention as the earlier notebooks. All saved artifacts are expected under the local `data/` directory of the project repository.

In [ ]:
import numpy as np
import torch
import pyro
import pyro.distributions as dist
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from pathlib import Path

np.random.seed(67)
torch.manual_seed(67)
pyro.set_rng_seed(67)

plt.style.use('ggplot')
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)
sns.set_context('talk')

print(f"pyro {pyro.__version__}, torch {torch.__version__}, numpy {np.__version__}")

## 1  Locate project data and load Phase 0 data

Phase 0 is still the source of truth for the raw cleaned tables: `songs_clean.csv` and `listens_clean.csv`. Phase 1 and Phase 3 save fitted model artifacts into subdirectories under `data/`, which this notebook uses for PPCs.

In [ ]:
def find_project_root(start=None, project_name="MBML---2026"):
    start = Path.cwd() if start is None else Path(start).resolve()
    for p in [start, *start.parents]:
        if p.name == project_name:
            return p
    fallback = Path.home() / "Desktop" / "mbml" / project_name
    if fallback.exists():
        return fallback
    return start

REPO_ROOT = find_project_root()
DATA_DIR = REPO_ROOT / 'data'

SONGS_CSV = DATA_DIR / 'songs_clean.csv'
LISTENS_CSV = DATA_DIR / 'listens_clean.csv'

assert SONGS_CSV.exists(), f"{SONGS_CSV} not found — run Phase 0 first."
assert LISTENS_CSV.exists(), f"{LISTENS_CSV} not found — run Phase 0 first."

df_songs_raw = pd.read_csv(SONGS_CSV)
df_listens_raw = pd.read_csv(LISTENS_CSV)

print("REPO_ROOT:", REPO_ROOT)
print("DATA_DIR :", DATA_DIR)
print(f"Songs loaded:   {len(df_songs_raw):,}")
print(f"Listens loaded: {len(df_listens_raw):,}")

## 2  Load Phase 1 mood-model outputs

The Phase 1 save cell should have written posterior samples or at least posterior medians for the mood model. Phase 4 uses those parameters to generate synthetic songs from the fitted mixed-likelihood model.

In [ ]:
PHASE1_OUT_DIR = DATA_DIR / 'phase1_processed'
assert PHASE1_OUT_DIR.exists(), (
    f"{PHASE1_OUT_DIR} not found. Run the Phase 1 save-contract cell first."
)

# Core arrays used for real-vs-fake comparisons.
X_cont_np = np.load(PHASE1_OUT_DIR / 'X_cont.npy')
X_key_np = np.load(PHASE1_OUT_DIR / 'X_key.npy')
X_ts_np = np.load(PHASE1_OUT_DIR / 'X_ts.npy')
X_mode_np = np.load(PHASE1_OUT_DIR / 'X_mode.npy')

# Metadata.
import json
with open(PHASE1_OUT_DIR / 'phase1_metadata.json', 'r') as f:
    phase1_meta = json.load(f)

CONT_COLS = phase1_meta['continuous_columns']
N_KEY = int(phase1_meta['N_KEY'])
N_TS = int(phase1_meta['N_TS'])
KEY_NAMES = phase1_meta.get('KEY_NAMES', [f'key={i}' for i in range(N_KEY)])
TS_LABELS = phase1_meta.get('TS_LABELS', [f'ts={i}' for i in range(N_TS)])
MOOD_NAMES = phase1_meta.get('MOOD_NAMES', [f'Mood {i}' for i in range(int(phase1_meta.get('K', 10)))])

# Posterior samples are preferred; posterior medians are a fallback.
posterior_samples_path = PHASE1_OUT_DIR / 'phase1_svi_posterior_samples.pt'
posterior_median_path = PHASE1_OUT_DIR / 'phase1_svi_median.pt'

if posterior_samples_path.exists():
    mood_posterior = torch.load(posterior_samples_path, map_location='cpu')
    print("Loaded Phase 1 SVI posterior samples.")
elif posterior_median_path.exists():
    median = torch.load(posterior_median_path, map_location='cpu')
    mood_posterior = {k: v.unsqueeze(0) for k, v in median.items()}
    print("Loaded Phase 1 SVI posterior median only; PPC will use point-parameter draws.")
else:
    raise FileNotFoundError(
        "Need either phase1_svi_posterior_samples.pt or phase1_svi_median.pt in data/phase1_processed."
    )

z_map = np.load(PHASE1_OUT_DIR / 'z_map.npy') if (PHASE1_OUT_DIR / 'z_map.npy').exists() else None

print("Phase 1 arrays:")
print("X_cont_np:", X_cont_np.shape)
print("X_key_np :", X_key_np.shape)
print("X_ts_np  :", X_ts_np.shape)
print("X_mode_np:", X_mode_np.shape)
print("Mood posterior sites:")
for k, v in mood_posterior.items():
    print(f"  {k:12s}: {tuple(v.shape)}")
if z_map is not None:
    print("z_map:", z_map.shape)

## 3  Load Phase 3 user-taste outputs

Phase 3 contains the fitted user-taste model. We load the posterior taste profiles, user activity bias, song popularity bias, and learned global calibration terms so that Phase 4 evaluates the **same likelihood** used in Phase 3.

In [ ]:
PHASE3_OUT_DIR = DATA_DIR / 'phase3_processed'
assert PHASE3_OUT_DIR.exists(), (
    f"{PHASE3_OUT_DIR} not found. Run the Phase 3 save-contract cell first."
)

theta_post = np.load(PHASE3_OUT_DIR / 'theta_post.npy')
entropy = np.load(PHASE3_OUT_DIR / 'entropy.npy') if (PHASE3_OUT_DIR / 'entropy.npy').exists() else None

# These are essential for using the refined Phase 3 listen model.
alpha_u_post = np.load(PHASE3_OUT_DIR / 'alpha_u_post.npy')
gamma_s_post = np.load(PHASE3_OUT_DIR / 'gamma_s_post.npy')

df_listens = pd.read_csv(PHASE3_OUT_DIR / 'df_listens_phase3_used.csv')
df_users = pd.read_csv(PHASE3_OUT_DIR / 'users_phase3.csv')

with open(PHASE3_OUT_DIR / 'phase3_metadata.json', 'r') as f:
    phase3_meta = json.load(f)

K = int(phase3_meta['K'])
U = int(phase3_meta['U'])
BASE_LOGIT = float(phase3_meta['BASE_LOGIT'])
TASTE_SCALE = float(phase3_meta['TASTE_SCALE'])
MOOD_NAMES = phase3_meta.get('MOOD_NAMES', MOOD_NAMES)

required_cols = {'user_idx', 'song_idx', 'mood', 'listened'}
assert required_cols.issubset(df_listens.columns), (
    f"df_listens_phase3_used.csv must contain {required_cols}; found {df_listens.columns.tolist()}"
)

print("Loaded Phase 3 outputs:")
print("theta_post  :", theta_post.shape)
print("alpha_u_post:", alpha_u_post.shape)
print("gamma_s_post:", gamma_s_post.shape)
print("df_listens  :", df_listens.shape)
print(f"K={K}, U={U}, BASE_LOGIT={BASE_LOGIT:.3f}, TASTE_SCALE={TASTE_SCALE:.3f}")

## 4  Posterior predictive song generator

For each posterior draw, we sample new mood assignments and then sample the observed audio variables from the fitted likelihoods. Continuous features are sampled from Normal distributions, key and time signature from Categorical distributions, and mode from a Bernoulli distribution.

In [ ]:
def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def get_param_draw(posterior, draw_idx):
    # Extract one posterior draw, cycling if fewer draws are available.
    out = {}
    for name, value in posterior.items():
        if not isinstance(value, torch.Tensor):
            value = torch.tensor(value)
        n_draws = value.shape[0]
        out[name] = value[draw_idx % n_draws]
    return out


def sample_synthetic_songs(param, n_songs, rng):
    pi = to_numpy(param['pi'])
    mu_cont = to_numpy(param['mu_cont'])
    sigma_cont = np.maximum(to_numpy(param['sigma_cont']), 1e-6)
    theta_key = to_numpy(param['theta_key'])
    theta_ts = to_numpy(param['theta_ts'])
    p_mode = np.clip(to_numpy(param['p_mode']), 1e-6, 1 - 1e-6)

    K_local = len(pi)
    pi = pi / pi.sum()

    z = rng.choice(K_local, size=n_songs, p=pi)

    x_cont = np.zeros((n_songs, mu_cont.shape[1]), dtype=np.float32)
    x_key = np.zeros(n_songs, dtype=int)
    x_ts = np.zeros(n_songs, dtype=int)
    x_mode = np.zeros(n_songs, dtype=int)

    for k in range(K_local):
        idx = np.where(z == k)[0]
        if len(idx) == 0:
            continue
        x_cont[idx] = rng.normal(mu_cont[k], sigma_cont[k], size=(len(idx), mu_cont.shape[1]))
        x_key[idx] = rng.choice(theta_key.shape[1], size=len(idx), p=theta_key[k] / theta_key[k].sum())
        x_ts[idx] = rng.choice(theta_ts.shape[1], size=len(idx), p=theta_ts[k] / theta_ts[k].sum())
        x_mode[idx] = rng.binomial(1, p_mode[k], size=len(idx))

    return z, x_cont, x_key, x_ts, x_mode


N_PPC = 100
N_FAKE_SONGS = 1000
rng = np.random.default_rng(67)

fake_mood_all = []
fake_cont_all = []
fake_key_all = []
fake_ts_all = []
fake_mode_all = []

for draw_idx in range(N_PPC):
    param = get_param_draw(mood_posterior, draw_idx)
    z_fake, cont_fake, key_fake, ts_fake, mode_fake = sample_synthetic_songs(param, N_FAKE_SONGS, rng)
    fake_mood_all.append(z_fake)
    fake_cont_all.append(cont_fake)
    fake_key_all.append(key_fake)
    fake_ts_all.append(ts_fake)
    fake_mode_all.append(mode_fake)

fake_mood_all = np.asarray(fake_mood_all)
fake_cont_all = np.asarray(fake_cont_all)
fake_key_all = np.asarray(fake_key_all)
fake_ts_all = np.asarray(fake_ts_all)
fake_mode_all = np.asarray(fake_mode_all)

print("Synthetic song arrays:")
print("fake_mood_all:", fake_mood_all.shape)
print("fake_cont_all:", fake_cont_all.shape)
print("fake_key_all :", fake_key_all.shape)
print("fake_ts_all  :", fake_ts_all.shape)
print("fake_mode_all:", fake_mode_all.shape)

## 5  PPC for continuous audio features

We compare the observed z-scored continuous features to synthetic features generated from the fitted mood model. Good posterior predictive fit means the fake and real histograms should broadly overlap.

In [ ]:
fake_cont_flat = fake_cont_all.reshape(-1, fake_cont_all.shape[-1])

fig, axes = plt.subplots(1, len(CONT_COLS), figsize=(18, 5))
if len(CONT_COLS) == 1:
    axes = [axes]

for d, (ax, col) in enumerate(zip(axes, CONT_COLS)):
    sns.histplot(X_cont_np[:, d], bins=50, stat='density', alpha=0.45, label='real', ax=ax)
    sns.histplot(fake_cont_flat[:, d], bins=50, stat='density', alpha=0.45, label='posterior predictive', ax=ax)
    ax.set_title(f'PPC: {col}')
    ax.set_xlabel('z-score')
    ax.legend()

plt.tight_layout()
plt.show()

feature_mean_real = X_cont_np.mean(axis=0)
feature_mean_fake = fake_cont_flat.mean(axis=0)
feature_std_real = X_cont_np.std(axis=0)
feature_std_fake = fake_cont_flat.std(axis=0)

feature_summary = pd.DataFrame({
    'feature': CONT_COLS,
    'real_mean': feature_mean_real,
    'fake_mean': feature_mean_fake,
    'mean_abs_diff': np.abs(feature_mean_real - feature_mean_fake),
    'real_std': feature_std_real,
    'fake_std': feature_std_fake,
    'std_abs_diff': np.abs(feature_std_real - feature_std_fake),
})

feature_summary.round(4)

## 6  PPC for categorical audio features

Key, time signature, and mode are discrete variables. We compare their observed empirical proportions to posterior predictive proportions and show 90% predictive intervals across posterior draws.

In [ ]:
def posterior_predictive_category_summary(fake_arr, n_categories):
    fake_arr = np.asarray(fake_arr)
    if fake_arr.ndim == 1:
        fake_arr = fake_arr[None, :]
    props = []
    for sample in fake_arr:
        sample = np.asarray(sample).astype(int).ravel()
        counts = np.bincount(sample, minlength=n_categories)[:n_categories]
        props.append(counts / max(len(sample), 1))
    props = np.stack(props)
    return props.mean(0), np.percentile(props, 5, axis=0), np.percentile(props, 95, axis=0)

key_mean, key_lo, key_hi = posterior_predictive_category_summary(fake_key_all, N_KEY)
ts_mean, ts_lo, ts_hi = posterior_predictive_category_summary(fake_ts_all, N_TS)
mode_mean, mode_lo, mode_hi = posterior_predictive_category_summary(fake_mode_all, 2)

real_key = np.bincount(X_key_np.astype(int), minlength=N_KEY)[:N_KEY] / len(X_key_np)
real_ts = np.bincount(X_ts_np.astype(int), minlength=N_TS)[:N_TS] / len(X_ts_np)
real_mode = np.bincount(X_mode_np.astype(int), minlength=2)[:2] / len(X_mode_np)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

x = np.arange(N_KEY)
axes[0].bar(x - 0.2, real_key, width=0.4, label='real', alpha=0.55)
axes[0].bar(x + 0.2, key_mean, width=0.4, label='posterior predictive', alpha=0.65)
axes[0].errorbar(x + 0.2, key_mean, yerr=[key_mean - key_lo, key_hi - key_mean], fmt='none')
axes[0].set_xticks(x)
axes[0].set_xticklabels(KEY_NAMES, rotation=45)
axes[0].set_title('PPC: key')
axes[0].legend()

x = np.arange(N_TS)
axes[1].bar(x - 0.2, real_ts, width=0.4, label='real', alpha=0.55)
axes[1].bar(x + 0.2, ts_mean, width=0.4, label='posterior predictive', alpha=0.65)
axes[1].errorbar(x + 0.2, ts_mean, yerr=[ts_mean - ts_lo, ts_hi - ts_mean], fmt='none')
axes[1].set_xticks(x)
axes[1].set_xticklabels(TS_LABELS, rotation=45)
axes[1].set_title('PPC: time signature')
axes[1].legend()

x = np.arange(2)
axes[2].bar(x - 0.2, real_mode, width=0.4, label='real', alpha=0.55)
axes[2].bar(x + 0.2, mode_mean, width=0.4, label='posterior predictive', alpha=0.65)
axes[2].errorbar(x + 0.2, mode_mean, yerr=[mode_mean - mode_lo, mode_hi - mode_mean], fmt='none')
axes[2].set_xticks(x)
axes[2].set_xticklabels(['minor', 'major'])
axes[2].set_title('PPC: mode')
axes[2].legend()

plt.tight_layout()
plt.show()

## 7  PPC for mood-frequency distribution

The fitted model should generate mood frequencies similar to the real MAP mood assignments from Phase 1. This is a direct check of the learned mixture weights $\pi$.

In [ ]:
mood_mean, mood_lo, mood_hi = posterior_predictive_category_summary(fake_mood_all, K)

if z_map is not None:
    real_mood = np.bincount(z_map.astype(int), minlength=K)[:K] / len(z_map)
else:
    # Fallback: use the mean posterior mixture weights if no MAP z was saved.
    pi_arr = to_numpy(mood_posterior['pi'])
    real_mood = pi_arr.mean(axis=0)
    real_mood = real_mood / real_mood.sum()

x = np.arange(K)
plt.figure(figsize=(14, 6))
plt.bar(x - 0.2, real_mood, width=0.4, label='real MAP moods', alpha=0.55)
plt.bar(x + 0.2, mood_mean, width=0.4, label='posterior predictive', alpha=0.65)
plt.errorbar(x + 0.2, mood_mean, yerr=[mood_mean - mood_lo, mood_hi - mood_mean], fmt='none')
plt.xticks(x, [f'{k}\n{MOOD_NAMES[k] if k < len(MOOD_NAMES) else f"Mood {k}"}' for k in range(K)], rotation=45, ha='right')
plt.ylabel('Proportion')
plt.title('PPC: mood-frequency distribution')
plt.legend()
plt.tight_layout()
plt.show()

mood_l1 = float(np.abs(real_mood - mood_mean).sum())
print(f"Mood-frequency L1 distance: {mood_l1:.3f}")

## 8  Listen probability calibration using Phase 3 likelihood

This is the key Phase 4 check for the user-taste extension. We compute predicted listen probabilities using the same learned Phase 3 formula:

$$\mathrm{logit}(p_{us}) = b + eta\,	heta_{u,z_s} + lpha_u + \gamma_s.$$

Then we compare predicted probabilities to observed listen frequencies in calibration bins.

In [ ]:
user_idx_t = torch.tensor(df_listens['user_idx'].values, dtype=torch.long)
song_idx_t = torch.tensor(df_listens['song_idx'].values, dtype=torch.long)
mood_idx_t = torch.tensor(df_listens['mood'].values, dtype=torch.long)
listened_t = torch.tensor(df_listens['listened'].values, dtype=torch.float32)

theta_t = torch.tensor(theta_post, dtype=torch.float32)
alpha_u_t = torch.tensor(alpha_u_post, dtype=torch.float32)
gamma_s_t = torch.tensor(gamma_s_post, dtype=torch.float32)

# Safety check: song_idx must index gamma_s.
max_song_idx = int(song_idx_t.max())
assert max_song_idx < len(gamma_s_t), (
    f"song_idx max is {max_song_idx}, but gamma_s_post length is {len(gamma_s_t)}. "
    "Phase 3 must save gamma_s aligned to df_listens['song_idx']."
)

with torch.no_grad():
    theta_selected = theta_t[user_idx_t, mood_idx_t]
    logits = BASE_LOGIT + TASTE_SCALE * theta_selected + alpha_u_t[user_idx_t] + gamma_s_t[song_idx_t]
    pred_prob = torch.sigmoid(logits).cpu().numpy()

obs_listen = listened_t.cpu().numpy()

plt.figure(figsize=(10, 6))
sns.histplot(pred_prob[obs_listen == 1], bins=40, stat='density', alpha=0.45, label='observed listened=1')
sns.histplot(pred_prob[obs_listen == 0], bins=40, stat='density', alpha=0.45, label='observed listened=0')
plt.xlabel('Predicted listen probability')
plt.title('Predicted listen probabilities by observed label')
plt.legend()
plt.tight_layout()
plt.show()

calib_df = pd.DataFrame({'pred_prob': pred_prob, 'observed': obs_listen})
calib_df['bin'] = pd.qcut(calib_df['pred_prob'], q=10, duplicates='drop')
calib = calib_df.groupby('bin', observed=True).agg(
    pred_mean=('pred_prob', 'mean'),
    empirical_rate=('observed', 'mean'),
    n=('observed', 'size'),
).reset_index()
calib['abs_error'] = (calib['pred_mean'] - calib['empirical_rate']).abs()
calib_mae = float(calib['abs_error'].mean())

plt.figure(figsize=(8, 8))
plt.plot([0, 1], [0, 1], 'k--', label='perfect calibration')
plt.scatter(calib['pred_mean'], calib['empirical_rate'], s=np.sqrt(calib['n']) * 2, label='bins')
for _, row in calib.iterrows():
    plt.text(row['pred_mean'], row['empirical_rate'], f"n={int(row['n'])}", fontsize=8)
plt.xlabel('Mean predicted probability')
plt.ylabel('Empirical listen frequency')
plt.title(f'Calibration plot — MAE={calib_mae:.3f}')
plt.legend()
plt.tight_layout()
plt.show()

print(calib[['pred_mean', 'empirical_rate', 'n', 'abs_error']].round(4).to_string(index=False))
print(f"\nCalibration MAE: {calib_mae:.4f}")

## 9  Listen-count posterior predictive check

Using the Phase 3 predicted probabilities, we simulate new listen outcomes for the same observed $(u,s)$ pairs. We then compare the distribution of positive listen counts per user between real and fake data.

In [ ]:
N_LISTEN_PPC = 100
rng = np.random.default_rng(67)

real_counts = df_listens.groupby('user_idx')['listened'].sum().reindex(range(U), fill_value=0).values
fake_counts_all = []

for _ in range(N_LISTEN_PPC):
    fake_l = rng.binomial(1, pred_prob).astype(int)
    fake_df = pd.DataFrame({'user_idx': df_listens['user_idx'].values, 'fake_l': fake_l})
    fake_counts = fake_df.groupby('user_idx')['fake_l'].sum().reindex(range(U), fill_value=0).values
    fake_counts_all.append(fake_counts)

fake_counts_all = np.stack(fake_counts_all)
fake_counts_flat = fake_counts_all.reshape(-1)

plt.figure(figsize=(12, 6))
sns.histplot(real_counts, bins=50, stat='density', alpha=0.45, label='real')
sns.histplot(fake_counts_flat, bins=50, stat='density', alpha=0.35, label='posterior predictive')
plt.xlabel('Positive listens per user')
plt.title('PPC: listen-count distribution per user')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Real listen counts: median={np.median(real_counts):.1f}, mean={np.mean(real_counts):.1f}")
print(f"Fake listen counts: median={np.median(fake_counts_flat):.1f}, mean={np.mean(fake_counts_flat):.1f}")

## 10  Save Phase 4 outputs

We save numeric summaries and calibration bins so they can be used directly in the final report. The plots are diagnostic; the CSV files are the reproducible outputs.

In [ ]:
PHASE4_OUT_DIR = DATA_DIR / 'phase4_processed'
PHASE4_OUT_DIR.mkdir(parents=True, exist_ok=True)

summary_rows = []
for d, col in enumerate(CONT_COLS):
    summary_rows.append({
        'check': 'continuous_feature',
        'name': col,
        'real_mean': float(feature_mean_real[d]),
        'fake_mean': float(feature_mean_fake[d]),
        'mean_abs_diff': float(abs(feature_mean_real[d] - feature_mean_fake[d])),
        'real_std': float(feature_std_real[d]),
        'fake_std': float(feature_std_fake[d]),
        'std_abs_diff': float(abs(feature_std_real[d] - feature_std_fake[d])),
    })

summary_rows.append({'check': 'mood_frequency', 'name': 'L1', 'value': float(mood_l1)})
summary_rows.append({'check': 'listen_calibration', 'name': 'MAE', 'value': float(calib_mae)})

df_ppc_summary = pd.DataFrame(summary_rows)
df_ppc_summary.to_csv(PHASE4_OUT_DIR / 'phase4_ppc_summary.csv', index=False)
calib.to_csv(PHASE4_OUT_DIR / 'phase4_calibration_bins.csv', index=False)
feature_summary.to_csv(PHASE4_OUT_DIR / 'phase4_feature_summary.csv', index=False)

np.save(PHASE4_OUT_DIR / 'real_mood_proportions.npy', real_mood)
np.save(PHASE4_OUT_DIR / 'fake_mood_proportions_mean.npy', mood_mean)
np.save(PHASE4_OUT_DIR / 'fake_mood_proportions_lo.npy', mood_lo)
np.save(PHASE4_OUT_DIR / 'fake_mood_proportions_hi.npy', mood_hi)
np.save(PHASE4_OUT_DIR / 'real_user_listen_counts.npy', real_counts)
np.save(PHASE4_OUT_DIR / 'fake_user_listen_counts.npy', fake_counts_all)

print("Saved Phase 4 outputs to:")
print(PHASE4_OUT_DIR)
for p in sorted(PHASE4_OUT_DIR.iterdir()):
    print('-', p.name)

## 11  Phase 4 gate summary

The gate is qualitative: real and fake feature histograms should overlap, mood frequencies should be close, and calibration should not show severe systematic bias. The summary below gives the key numerical checks to cite in the report.

In [ ]:
print("=" * 70)
print("PHASE 4 GATE SUMMARY — posterior predictive checks")
print("=" * 70)
print(f"Posterior predictive samples: {N_PPC}")
print(f"Synthetic songs per sample:   {N_FAKE_SONGS:,}")
print(f"Listen PPC replicates:        {N_LISTEN_PPC}")
print()
for _, row in feature_summary.iterrows():
    print(
        f"{row['feature']:<10s} "
        f"mean diff={row['mean_abs_diff']:.3f}  "
        f"std diff={row['std_abs_diff']:.3f}"
    )
print()
print(f"Mood-frequency L1 distance:   {mood_l1:.3f}")
print(f"Calibration MAE:              {calib_mae:.3f}")
print()
print("Interpretation:")
print("- Good PPC: real and fake histograms overlap, mood frequencies are close, calibration points are near diagonal.")
print("- Weak PPC: systematic shifts in feature histograms, missing rare categories, or calibration far from diagonal.")